# SHAP Explanation for Eval Model

This notebook loads the model specified in `backend/configs/eval.yaml`, runs preprocessing identical to `eval.py`, and computes SHAP feature importance.

In [4]:
import sys
import numpy as np
import sys
import pandas as pd
import torch
import shap
import pickle
from pathlib import Path
from omegaconf import OmegaConf
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from utils.seed import set_seed
from utils.device import get_device
from data.preprocessing import get_feature_columns, create_windows, clean_features
from model.bilstm import BiLSTM

# Config
cfg = OmegaConf.load(Path('../configs/eval.yaml'))
model_path = Path(cfg.model_path)
run_dir = model_path.parent

set_seed(cfg.seed)
device = get_device(cfg.device)
print('Loading artifacts from:', run_dir)


Loading artifacts from: outputs\2026-02-05\08-32-56


In [5]:
# Load scaler + label encoder + checkpoint
with open(run_dir / 'scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open(run_dir / 'label_encoder.pkl', 'rb') as f:
    encoder = pickle.load(f)

checkpoint = torch.load(model_path, map_location=device)


FileNotFoundError: [Errno 2] No such file or directory: 'outputs\\2026-02-05\\08-32-56\\scaler.pkl'

In [ ]:
# Load evaluation data (P2)
PROJECT_ROOT = Path('.')
DATA_DIR = PROJECT_ROOT / 'output_data'

df_boning = pd.read_csv(DATA_DIR / 'P2_boning.csv')
df_slicing = pd.read_csv(DATA_DIR / 'P2_slicing.csv')
df = pd.concat([df_boning, df_slicing], ignore_index=True)

# Filter sensor
df = df[df['sensor_type'] == cfg.data.sensor_type]

# Optional video suffix
if cfg.data.video_suffix is not None:
    df = df[df['video_id'].str.endswith(cfg.data.video_suffix)]

assert df['activity_type'].nunique() > 1, 'Only one class left after filtering'
print('Filtered rows:', len(df))


In [ ]:
# Preprocessing (IDENTICAL to training)
feature_cols = get_feature_columns(df)

X, y = create_windows(df, feature_cols, cfg.data.window_size)
X = clean_features(X)

# Normalize using training scaler
X = scaler.transform(X.reshape(-1, X.shape[-1])).reshape(X.shape)
y = encoder.transform(y)

print('X shape:', X.shape, 'y shape:', y.shape)


In [ ]:
# Build model
model = BiLSTM(
    input_size=X.shape[2],
    hidden_size=cfg.model.hidden_size,
    num_classes=len(encoder.classes_),
    num_layers=cfg.model.num_layers
).to(device)

state_dict = checkpoint.get('model_state_dict', checkpoint)
model.load_state_dict(state_dict)
model.eval()

print('Model loaded.')


In [ ]:
# SHAP computation
shap_sample = min(200, len(X))
shap_background = min(100, len(X))

rng = np.random.default_rng(cfg.seed)
sample_idx = rng.choice(len(X), size=shap_sample, replace=False)
bg_idx = rng.choice(len(X), size=shap_background, replace=False)

X_sample = torch.tensor(X[sample_idx], dtype=torch.float32, device=device)
X_bg = torch.tensor(X[bg_idx], dtype=torch.float32, device=device)

explainer = shap.DeepExplainer(model, X_bg)
shap_values = explainer.shap_values(X_sample)

shap_dir = run_dir / 'shap'
shap_dir.mkdir(parents=True, exist_ok=True)

if isinstance(shap_values, list):
    shap_stack = np.stack(shap_values, axis=0)
    shap_abs = np.mean(np.abs(shap_stack), axis=(0, 1, 2))
    for i, vals in enumerate(shap_values):
        np.save(shap_dir / f'shap_values_class_{i}.npy', vals)
else:
    shap_abs = np.mean(np.abs(shap_values), axis=(0, 1))
    np.save(shap_dir / 'shap_values.npy', shap_values)

importance = (
    pd.DataFrame({'feature': feature_cols, 'mean_abs_shap': shap_abs})
    .sort_values('mean_abs_shap', ascending=False)
)

importance.to_csv(shap_dir / 'shap_feature_importance.csv', index=False)
print('SHAP saved to', shap_dir)


In [ ]:
# Show top features
importance.head(20)
